# 01. 인공지능기본법 지원데스크 사례집 전처리

**수정 사항**: Q/A 분리를 마지막 `?` 기준으로 변경
- 기존: `Q...A` 정규식 (non-greedy, 첫 번째 A에서 잘림)
- 수정: 전체 텍스트에서 **마지막 `?`** 위치를 기준으로 질문/답변 분리

In [ ]:
#!pip install pdfplumber -q

In [ ]:
import pdfplumber, re, json
from pathlib import Path

PDF_PATH = "/Users/gimseoyeon/Downloads/AICPP/data/processed/인공지능기본법_지원데스크_사례집.pdf"
OUTPUT   = "/Users/gimseoyeon/Downloads/AICPP/data/processed/사례집_chunks.jsonl"

assert Path(PDF_PATH).exists(), f"PDF 없음: {PDF_PATH}"
Path(OUTPUT).parent.mkdir(parents=True, exist_ok=True)
print("PDF 확인:", PDF_PATH)

## 1. 전체 텍스트 추출

In [ ]:
def extract_pages(pdf_path):
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            t = page.extract_text()
            if t and t.strip():
                pages.append({"page": i+1, "text": t.strip()})
    print(f"{len(pages)}개 페이지 추출")
    return pages

pages = extract_pages(PDF_PATH)
print(pages[0]["text"][:200])


FileNotFoundError: [Errno 2] No such file or directory: '인공지능기본법_지원데스크_사례집.pdf'

## 2. 텍스트 클리닝

In [ ]:
def clean(text):
    # 제목·페이지번호 반복 제거
    text = re.sub(r'인공지능기본법 지원데스크 사례집\s*', '', text)
    text = re.sub(r'\[20선의 질답 사례로 배우는 기업 실무 가이드\]\s*', '', text)
    text = re.sub(r'^\s*\d{1,3}\s*$', '', text, flags=re.MULTILINE)
    # 줄바꿈으로 쪼개진 한글 단어 이어붙이기
    text = re.sub(r'([가-힣])\n([가-힣])', r'\1\2', text)
    text = re.sub(r'([가-힣])\n([가-힣])', r'\1\2', text)  # 2회 적용
    # 공백 정리
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

for p in pages:
    p["clean"] = clean(p["text"])

print(pages[3]["clean"][:300])


## 3. Q/A 분리 — **마지막 `?` 기준**

In [ ]:
def split_qa(raw_body):
    """
    수정된 Q/A 분리 함수.
    규칙: 마지막 '?' 위치까지가 질문, 그 이후가 답변.
    """
    # 줄바꿈으로 쪼개진 단어 복원
    text = re.sub(r'([가-힣a-zA-Z])\n([가-힣])', r'\1\2', raw_body)
    text = re.sub(r'\s+', ' ', text).strip()

    # Q 마커 제거
    text = re.sub(r'^Q[.:\s]+', '', text)

    # 마지막 '?' 위치 탐색
    idx = text.rfind('?')
    if idx == -1:
        # '?'가 없으면 'A' 마커 기반 분리 시도
        am = re.search(r'\bA[.:\s]+(.+)', text, re.DOTALL)
        if am:
            q = text[:am.start()].strip()
            a = am.group(1).strip()
        else:
            q, a = text, ""
        return q, a

    question = text[:idx+1].strip()
    answer_raw = text[idx+1:].strip()

    # 답변 앞 페이지번호·A 마커 제거
    answer_raw = re.sub(r'^\s*\d{1,3}\s*', '', answer_raw)
    answer_raw = re.sub(r'^A[.:\s]+', '', answer_raw).strip()
    answer = re.sub(r'\s+', ' ', answer_raw).strip()

    return question, answer


## 4. 사례 추출 및 JSONL 저장

In [ ]:
# 전체 텍스트 합산
full_text = "\n\n".join(p["clean"] for p in pages)

# 사례 패턴: <사례 N> 또는 사례 N:
CASE_PAT = re.compile(
    r'(?:<사례\s*|사례\s*)(\d+)[>\s:·]*([^\n]*)\n(.*?)(?=(?:<사례\s*|사례\s*)\d+|$)',
    re.DOTALL
)

# 유형 구분 (사례번호 → 유형)
TYPE_MAP = {
    range(1,6):  ("유형1","고영향 AI 해당 여부"),
    range(6,10): ("유형2","투명성·설명의무"),
    range(10,15):("유형3","이용자 보호"),
    range(15,21):("유형4","사업자 책무"),
}
def get_type(num):
    for r,(code,name) in TYPE_MAP.items():
        if num in r:
            return code, name
    return "기타", "기타"

chunks = []
for m in CASE_PAT.finditer(full_text):
    num    = int(m.group(1))
    title  = m.group(2).strip()
    body   = m.group(3).strip()

    q, a = split_qa(body)
    type_code, type_name = get_type(num)

    chunk = {
        "chunk_id":  f"사례집_{num:03d}",
        "유형번호":  type_code,
        "유형명":    type_name,
        "사례번호":  num,
        "제목":      title,
        "질문":      q,
        "답변":      a,
        "content":   f"[질문] {q}\n[답변] {a}",
        "출처":      "인공지능기본법 지원데스크 사례집",
        "청크유형":  "QA사례",
    }
    chunks.append(chunk)

print(f"총 {len(chunks)}개 사례 추출")
for c in chunks[:3]:
    print(f"  사례{c['사례번호']}: 질문={c['질문'][:50]}...")


In [ ]:
with open(OUTPUT, "w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")
print(f"저장 완료: {OUTPUT}")


## 5. 검증: 질문이 '?'로 끝나는지 확인

In [ ]:
errors = [c for c in chunks if not c["질문"].endswith("?")]
print(f"'?'로 끝나지 않는 사례: {len(errors)}개")
for e in errors[:5]:
    print(f"  사례{e['사례번호']}: {e['질문'][-60:]}")
